In [1]:
from os import environ
from pathlib import Path
import sys
environ["OPENBLAS_NUM_THREADS"] = "1"
_project_root = Path.cwd().resolve()
sys.path.insert(0, str(_project_root.parents[0]))
import matplotlib.pyplot as plt
import numpy as onp

import dataclasses as dcls

# Local Imports
import src.symmetry as sym
from src.categorization import print_symmetry_operations

NDArray = onp.ndarray

In [2]:

stat_arr_field = lambda arr: dcls.field(default_factory=lambda: arr)

def static_col_field(*xs):
	return stat_arr_field(onp.array([*xs]))


@dcls.dataclass(frozen=True)
class CanonBasis3:
	e1: NDArray = stat_arr_field(onp.array([1, 0, 0]))
	e2: NDArray = stat_arr_field(onp.array([0, 1, 0]))
	e3: NDArray = stat_arr_field(onp.array([0, 0, 1]))



cmap = plt.get_cmap('afmhot').copy()
cmap.set_bad(color='whitesmoke')

B = CanonBasis3()

In [3]:
print_symmetry_operations(sym.tetrahedral_group())

**e**

<IPython.core.display.Math object>

---

**a3**

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

---

**b2**

<IPython.core.display.Math object>

---

In [ ]:
ndim = 3
_0 = onp.zeros(ndim)
sym_ops = [sym.identity(ndim)]


gl_ref = sym.glide_reflection
ref = sym.reflection
m = [
	ref(B.e1),
	ref(B.e2),
	ref(B.e3),
]


qRM = sym.rotation3d
rot_order = 3
dθ = 2 * onp.pi / rot_order
θs = onp.arange(1, rot_order) * dθ
# excluding identity
curried_qRM = lambda axis: lambda θ: qRM(θ, axis)
rot_3fold_1 = list(map(curried_qRM(B.e1 + B.e2 + B.e3), θs))
rot_3fold_2 = list(map(curried_qRM(B.e1 - B.e2 + B.e3), θs))
rot_3fold_3 = list(map(curried_qRM(- B.e1 + B.e2 + B.e3), θs))
rot_3fold_4 = list(map(curried_qRM(- B.e1 - B.e2 + B.e3), θs))

rot_order = 2
dθ = 2 * onp.pi / rot_order
θs = onp.arange(1, rot_order) * dθ
rot_2fold_x = list(map(curried_qRM(B.e1), θs))
rot_2fold_y = list(map(curried_qRM(B.e2), θs))
rot_2fold_z = list(map(curried_qRM(B.e3), θs))

sym_ops += m
sym_ops += rot_2fold_x + rot_2fold_y + rot_2fold_z
sym_ops += rot_3fold_1 + rot_3fold_2 + rot_3fold_3 + rot_3fold_4

sym_ops.append(sym.identity(ndim, inversion=True))

# TODO investigate numerical error here...
# TODO frequently get 2e-16 errors in matrix
_sym_ops = []
for op in sym_ops:
	op = op.replace(matrix=onp.where(onp.abs(op.matrix) < 1e-10, 0, op.matrix))
	_sym_ops.append(op)

print_symmetry_operations(_sym_ops)

**e**

<IPython.core.display.Math object>

---

**s**

<IPython.core.display.Math object>

...

<IPython.core.display.Math object>

---

**e**

<IPython.core.display.Math object>

---

In [ ]:
N = 256
a = 64

def gaussian(mu, x):
	return onp.random.normal(x - mu)

grid = onp.zeros((N, N))
for i in range(-N, N, a):
	for j in range(-N, N, a):
		grid[i, j] = 1

def translate_image(img, shift_x, shift_y):
    return onp.roll(onp.roll(img, shift_x, axis=0), shift_y, axis=1)

lattice_original = grid.copy()
lattice_lattice = translate_image(grid, a, 0) 
lattice_nonlattice = translate_image(grid, a // 2, 0) 

def diffraction_pattern(img):
	F = onp.fft.fft2(img)
	F_shifted = onp.fft.fftshift(F)
	# intensity 
	return onp.abs(F_shifted) ** 2


diff_original = diffraction_pattern(lattice_original)
diff_lattice = diffraction_pattern(lattice_lattice)
diff_nonlattice = diffraction_pattern(lattice_nonlattice)


fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(lattice_original, cmap='gray', interpolation='nearest')
axes[0, 0].set_title("Original Lattice")
axes[0, 1].imshow(lattice_lattice, cmap='gray', interpolation='nearest')
axes[0, 1].set_title("Translation by Lattice Vector")
axes[0, 2].imshow(lattice_nonlattice, cmap='gray', interpolation='nearest')
axes[0, 2].set_title("Translation by Non-Lattice Vector")

axes[1, 0].imshow(onp.log(diff_original + 1), cmap='inferno', interpolation='nearest')
axes[1, 0].set_title("Diffraction: Original")
axes[1, 1].imshow(onp.log(diff_lattice + 1), cmap='inferno', interpolation='nearest')
axes[1, 1].set_title("Diffraction: Lattice Translation")
axes[1, 2].imshow(onp.log(diff_nonlattice + 1), cmap='inferno', interpolation='nearest')
axes[1, 2].set_title("Diffraction: Non-Lattice Translation")

for ax in axes.flatten():
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()
